# Train a TBK SAE on Gemma 3 270M

This notebook trains a custom sparse autoencoder on `google/gemma-3-270m-it`, attached near the middle of the transformer blocks. It mixes roughly 2M tokens from a Pile-style dataset with the local `samples/questions_train.txt` data.

The SAE uses top-bottom-k activation (`tbk`), which keeps the largest-magnitude feature pre-activations and preserves their signs.

In [45]:
from pathlib import Path
import random
import sys

import torch
from datasets import load_dataset

REQUESTED_DEVICE = "cpu"  # Options: "auto", "cpu", "cuda", "cuda:0", "mps", ...


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "trainable_sae.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from trainable_sae import (
    HookPointSpec,
    SAEConfig,
    SAEConnector,
    TrainableSAE,
    load_hooked_transformer,
    resolve_device,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = resolve_device(REQUESTED_DEVICE)
dtype = torch.bfloat16 if device.startswith("cuda") else torch.float32
device, dtype

('cpu', torch.float32)

## Configuration

The hook layer is computed from the loaded model so it stays in the middle even if the model config changes. For Gemma 3 270M this should land around the center block.

In [46]:
MODEL_NAME = "google/gemma-3-270m-it"
MODEL_NAME = "pythia-70m"
PILE_DATASET = "NeelNanda/pile-10k"
QUESTIONS_PATH = PROJECT_ROOT / "samples/questions_train.txt"

PILE_TOKEN_BUDGET = 2_000_000
QUESTION_REPEATS = 3
BATCH_TOKENS = 4096
CONTEXT_SIZE = 64
EXPANSION_FACTOR = 16
TOP_K = 50
LR = 2e-4
MAX_STEPS = None  # Set to a small integer like 10 for a quick smoke test.

SAVE_DIR = PROJECT_ROOT / "custom_saes/gemma3_270m_mid_resid_post_tbk_pile2m_questions"

## Load Gemma 3 270M

You may need to be logged into Hugging Face and have accepted the Gemma license before this cell can download the model.

In [47]:
model = load_hooked_transformer(
    MODEL_NAME,
    device=device,
    dtype=dtype,
)

mid_layer = model.cfg.n_layers // 2
hook_point = HookPointSpec(layer=mid_layer, site="resid_post").name
d_in = model.cfg.d_model

print(f"model:      {MODEL_NAME}")
print(f"layers:     {model.cfg.n_layers}")
print(f"hook point: {hook_point}")
print(f"d_in:       {d_in}")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 7030.28it/s]


Loaded pretrained model pythia-70m into HookedTransformer
Moving model to device:  cpu
model:      pythia-70m
layers:     6
hook point: blocks.3.hook_resid_post
d_in:       512


## Build the TBK SAE and connector

In [48]:
sae_cfg = SAEConfig(
    d_in=d_in,
    d_sae=d_in * EXPANSION_FACTOR,
    activation="tbk",
    k=TOP_K,
    lr=LR,
    l1_coefficient=0.0,  # TBK already enforces exactly k active features.
    dtype="float32",    # Keep SAE training in fp32 even if the model is bf16.
    device=device,
    metadata={
        "model_name": MODEL_NAME,
        "hook_point": hook_point,
        "pile_dataset": PILE_DATASET,
        "pile_token_budget": PILE_TOKEN_BUDGET,
        "questions_path": str(QUESTIONS_PATH),
    },
)

sae = TrainableSAE(sae_cfg, device=device)
connector = SAEConnector(model=model, sae=sae, hook_point=hook_point, device=device)
optimizer = torch.optim.AdamW(sae.parameters(), lr=sae_cfg.lr)

print(f"SAE: {sae_cfg.d_in} -> {sae_cfg.d_sae}, activation={sae_cfg.activation}, k={sae_cfg.k}")

Moving model to device:  cpu
SAE: 512 -> 8192, activation=tbk, k=50


## Prepare training text

`NeelNanda/pile-10k` is the same Pile-style source used elsewhere in this repo. The helper below takes approximately 2M model tokens from it, then mixes in the local generated question data.

In [ ]:
def read_questions(path: Path, repeats: int = 1):
    lines = [line.strip() for line in path.read_text().splitlines() if line.strip()]
    return lines * repeats


def pile_texts_until_token_budget(dataset_name: str, token_budget: int):
    dataset = load_dataset(dataset_name, split="train", streaming=True)
    total_tokens = 0
    for row in dataset:
        text = row.get("text", "")
        if not text.strip():
            continue
        total_tokens += model.to_tokens(text, truncate=True).numel()
        yield text
        if total_tokens >= token_budget:
            break
    print(f"Collected about {total_tokens:,} Pile tokens.")


question_texts = read_questions(QUESTIONS_PATH, repeats=QUESTION_REPEATS)
pile_texts = list(pile_texts_until_token_budget(PILE_DATASET, PILE_TOKEN_BUDGET))

training_texts = pile_texts + question_texts
random.shuffle(training_texts)

print(f"Pile examples:     {len(pile_texts):,}")
print(f"Question examples: {len(question_texts):,}")
print(f"Total examples:    {len(training_texts):,}")

## Activation batching

This harvests activations from the chosen middle block and flattens `[batch, seq, d_model]` into token rows for SAE training.

In [50]:
def activation_batches(texts, batch_tokens=BATCH_TOKENS, context_size=CONTEXT_SIZE):
    buffer = []
    buffered = 0

    for text in texts:
        tokens = model.to_tokens(text, truncate=True)
        if tokens.shape[1] > context_size:
            tokens = tokens[:, :context_size]

        acts = connector.collect_activations(tokens).reshape(-1, d_in)
        buffer.append(acts)
        buffered += acts.shape[0]

        if buffered >= batch_tokens:
            joined = torch.cat(buffer, dim=0)
            yield joined[:batch_tokens]
            remainder = joined[batch_tokens:]
            buffer = [remainder] if remainder.numel() else []
            buffered = remainder.shape[0] if remainder.numel() else 0

    if buffer:
        yield torch.cat(buffer, dim=0)

## Train

In [ ]:
metrics = []

for step, batch in enumerate(activation_batches(training_texts), start=1):
    batch = batch.to(device=device, dtype=torch.float32)
    step_metrics = sae.training_step(batch, optimizer)
    metrics.append(step_metrics)

    if step == 1 or step % 10 == 0:
        print(
            f"step {step:04d} | "
            f"loss={step_metrics['loss']:.5f} | "
            f"mse={step_metrics['mse']:.5f} | "
            f"l1={step_metrics['l1']:.5f}"
        )

    if MAX_STEPS is not None and step >= MAX_STEPS:
        break

print(f"Finished {len(metrics)} optimization steps.")

## Save and smoke-test insertion

In [ ]:
SAVE_DIR.mkdir(parents=True, exist_ok=True)
sae.save(SAVE_DIR)
print(f"Saved SAE to {SAVE_DIR}")

prompt = "Sarah put the book in the kitchen. Answer in one word: where is the book?"
tokens = model.to_tokens(prompt)

with torch.no_grad():
    logits = connector.run_with_sae(tokens, mode="reconstruct")

print(logits.shape)
print("latest features:", connector.latest_features.shape if connector.latest_features is not None else None)

In [ ]:
EVAL_PROMPT = "Sarah put the book in the kitchen. Answer in one word: where is the book?"
MAX_NEW_TOKENS = 10
TEMPERATURE = 0.0

generate_kwargs = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    verbose=False,
)

def clean_generation(text: str) -> str:
    return text.split("<end_of_turn>")[0].strip()

tokens = model.to_tokens(EVAL_PROMPT).to(device)
prompt_len = tokens.shape[1]

model.reset_hooks()
with torch.no_grad():
    normal_tokens = model.generate(tokens, **generate_kwargs)
normal_output = clean_generation(model.to_string(normal_tokens[0, prompt_len:]))

with torch.no_grad():
    sae_output = clean_generation(
        connector.generate_with_sae(EVAL_PROMPT, mode="reconstruct", **generate_kwargs)
    )

print("Prompt:")
print(EVAL_PROMPT)
print("\nNormal model:")
print(normal_output)
print("\nWith SAE inserted:")
print(sae_output)
print("\nLatest SAE feature tensor:", connector.latest_features.shape if connector.latest_features is not None else None)
